# Modul 12: Unüberwachtes Lernen und Text als Merkmale | Übungen

## Überblick

Sie vergleichen Clusterverfahren, bewerten Cluster vorsichtig, erkennen Anomalien und nutzen wenige Labels mit semi-supervised Lernen. Danach bereiten Sie eine kleine deutsche Textsammlung auf, erzeugen Sparse-Merkmale und trainieren reproduzierbare Textklassifikationspipelines.

**Zugehörige Vorlesungen**

- **Unüberwacht lernen**
- **Text als Merkmale**

## Lernziele

Nach der Bearbeitung können Sie:

- KMeans, MiniBatchKMeans, hierarchisches Clustering, DBSCAN und GaussianMixture passend zu Datenformen vergleichen.
- Anomalieerkennung und LabelSpreading auf skalierten Daten anwenden und Unsicherheit dokumentieren.
- Textdaten bereinigen, mit Count- und TF-IDF-Merkmalen darstellen und mit linearen Modellen klassifizieren.

## Geprüfte Fähigkeiten

- Clustering, Silhouette, Adjusted Rand Index und probabilistische Zuordnungen
- IsolationForest und semi-supervised LabelSpreading
- CountVectorizer, TfidfVectorizer, Sparse-Matrizen und Textpipelines

## Hinweise zur Bearbeitung

Dieses Notebook dient als praktische Übung und Lernstandskontrolle. Führen Sie zuerst die Einrichtungszelle aus und bearbeiten Sie danach die Aufgaben in der angegebenen Reihenfolge. Die vorgesehenen Arbeitsbereiche sind deutlich markiert.

- **Erwarteter Schwierigkeitsgrad:** fortgeschritten
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle erzeugt zwei kleine geometrische Datensätze und eine lokale deutsche Textsammlung mit drei Klassen. Es sind keine externen Downloads oder Dateien erforderlich.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

X_blobs, y_blobs = make_blobs(
    n_samples=360,
    centers=[(-4, -1), (0, 3), (4, -1)],
    cluster_std=[0.75, 1.0, 0.8],
    random_state=RANDOM_SEED,
)
X_moons, y_moons = make_moons(n_samples=360, noise=0.08, random_state=RANDOM_SEED)

texte = [
    "Das Team gewann nach einer starken zweiten Halbzeit.",
    "Der Trainer lobte die Abwehr und den schnellen Angriff.",
    "Das Fußballspiel endete nach Verlängerung unentschieden.",
    "Die Läuferin stellte beim Wettkampf einen neuen Rekord auf.",
    "Der Torwart hielt den entscheidenden Elfmeter.",
    "Im Finale überzeugte die Mannschaft mit ruhigem Passspiel.",
    "Ein neues Softwareupdate verbessert die Datensicherheit.",
    "Der Prozessor arbeitet schneller und benötigt weniger Energie.",
    "Forschende testen einen kompakten Roboter für die Produktion.",
    "Die App verarbeitet Bilder nun direkt auf dem Smartphone.",
    "Das Netzwerk erkennt ungewöhnliche Zugriffe automatisch.",
    "Die neue Python-Bibliothek vereinfacht die Datenanalyse.",
    "Die Stadt eröffnet einen neuen Park mit vielen Bäumen.",
    "Der Zugverkehr wird wegen Bauarbeiten am Wochenende geändert.",
    "Die Schule plant zusätzliche Kurse für das kommende Jahr.",
    "Im Krankenhaus wurde eine neue Ambulanz eröffnet.",
    "Der Gemeinderat diskutiert günstigere Wohnungen.",
    "Die Bibliothek verlängert ihre Öffnungszeiten im Sommer.",
] * 3
labels = (["Sport"] * 6 + ["Technik"] * 6 + ["Alltag"] * 6) * 3

text_daten = pd.DataFrame({"Text": texte, "Kategorie": labels})
# Kleine Variationen verhindern vollständig identische Duplikate.
text_daten["Text"] = [f"{text} Hinweis {i % 3}." for i, text in enumerate(text_daten["Text"])]

print("Blob-Daten:", X_blobs.shape)
print("Moon-Daten:", X_moons.shape)
print("Textdaten:", text_daten.shape)

### Aufgabe 1: KMeans, MiniBatch und hierarchisches Clustering vergleichen

Standardisieren Sie `X_blobs` und passen Sie KMeans, MiniBatchKMeans und AgglomerativeClustering jeweils mit drei Clustern an. Berechnen Sie für jede Zuordnung Silhouette und Adjusted Rand Index gegenüber den nur zur nachträglichen Kontrolle bereitstehenden wahren Labels `y_blobs`.

Visualisieren Sie die drei Zuordnungen. Verwenden Sie die wahren Labels nicht beim Anpassen der Clusterverfahren.

In [ ]:
from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum ist der ARI in einer echten unbeschrifteten Aufgabe oft nicht verfügbar?

### Aufgabe 2: Komplexe Formen mit DBSCAN und GaussianMixture untersuchen

Standardisieren Sie `X_moons`. Passen Sie KMeans mit zwei Clustern, DBSCAN und ein GaussianMixture-Modell mit zwei Komponenten an. Visualisieren Sie die Zuordnungen und geben Sie für DBSCAN die Anzahl erkannter Cluster und Rauschpunkte aus.

Lassen Sie das GaussianMixture-Modell zusätzlich für die ersten fünf Punkte Komponentenwahrscheinlichkeiten ausgeben.

In [ ]:
from sklearn.cluster import DBSCAN, KMeans
from sklearn.mixture import GaussianMixture

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum kann KMeans bei halbmondförmigen Clustern scheitern?

### Aufgabe 3: Anomalien erkennen und fachlich prüfen

Erweitern Sie `X_blobs` um acht weit entfernte künstliche Punkte. Skalieren Sie die erweiterten Daten und passen Sie einen IsolationForest mit einem erwarteten Anomalieanteil von ungefähr zwei Prozent an.

Geben Sie die erkannten Anomalieindizes und deren Werte aus. Visualisieren Sie normale und auffällige Punkte und prüfen Sie, wie viele der acht künstlichen Punkte erkannt wurden.

In [ ]:
from sklearn.ensemble import IsolationForest

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum ist eine Anomaliekennzeichnung noch keine fachliche Fehlerdiagnose?

### Aufgabe 4: Mit wenigen Labels lernen

Verwenden Sie die standardisierten Blob-Daten. Lassen Sie pro wahrer Klasse nur drei zufällig ausgewählte Labels sichtbar und setzen Sie alle anderen Zielwerte auf `-1`. Trainieren Sie `LabelSpreading` und berechnen Sie die Genauigkeit der übertragenen Labels ausschließlich auf den zuvor unbeschrifteten Punkten.

Geben Sie außerdem die fünf unbeschrifteten Punkte mit der höchsten Unsicherheit aus. Nutzen Sie dazu die größte Klassenwahrscheinlichkeit als Sicherheitsmaß.

In [ ]:
from sklearn.semi_supervised import LabelSpreading
from sklearn.metrics import accuracy_score

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum sollten übertragene Labels nicht wie sichere menschliche Annotationen behandelt werden?

### Aufgabe 5: Count-, TF-IDF- und Hashing-Merkmale untersuchen

Schreiben Sie eine einfache Bereinigungsfunktion für deutsche Texte, die Kleinschreibung anwendet, technische Ziffern entfernt, Satzzeichen durch Leerzeichen ersetzt und Mehrfachleerzeichen reduziert.

Erzeugen Sie anschließend für die bereinigten Texte:

1. Count-Merkmale mit Unigrammen und Bigrammen,
2. TF-IDF-Merkmale mit denselben n-Grammen,
3. Hashing-Merkmale mit 64 Dimensionen.

Geben Sie Form, Typ und Dichte der drei Matrizen aus. Zeigen Sie außerdem die zehn häufigsten Count-Merkmale.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Welchen wichtigen Nachteil hat Hashing gegenüber CountVectorizer und TfidfVectorizer?

### Aufgabe 6: Integrationsaufgabe: Textpipelines vergleichen und Fehler analysieren

Teilen Sie die Textdaten stratifiziert in Training und Test. Vergleichen Sie drei vollständige Pipelines:

- CountVectorizer plus MultinomialNB,
- TfidfVectorizer plus LogisticRegression,
- TfidfVectorizer plus LinearSVC.

Nutzen Sie dieselbe Bereinigungsfunktion als `preprocessor`. Berechnen Sie Accuracy und Macro-F1. Wählen Sie nach Macro-F1 das beste Modell, erstellen Sie eine Konfusionsmatrix und geben Sie alle falsch klassifizierten Testtexte mit wahrer und vorhergesagter Klasse aus.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

# ============================================================
# IHRE LÖSUNG HIER
# ============================================================

> **Ihre Antwort:**
>
> Warum sollte bei mehreren Klassen zusätzlich zur Accuracy Macro-F1 betrachtet werden?

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?